# Evaluation: Reference-Based vs VLM Baseline

This notebook evaluates whether the reference-based size estimates outperform the no-reference VLM baseline.

Inputs:
- Ground truth: `data/cs543_data_collection_plan_updated.csv`
- VLM baseline JSON files: `results/vlm_baseline/*_vlm_baseline.json`
- Reference-based results: `results/reference_based_estimation.csv`

The key paired question is: for the same photo, is the reference-based error lower than the VLM baseline error?

In [ ]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None
    plt.style.use("ggplot")

try:
    from scipy import stats
except ImportError:
    stats = None
    warnings.warn("scipy is not installed; paired t-tests and Wilcoxon tests will be skipped.")

# Keep notebook tables readable when showing joined metadata and metrics.
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# Let the notebook run from either the repository root or the notebooks/ directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists() and (REPO_ROOT.parent / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

GT_PATH = REPO_ROOT / "data" / "cs543_data_collection_plan_updated.csv"
BASELINE_DIR = REPO_ROOT / "results" / "vlm_baseline"
REFERENCE_PATH = REPO_ROOT / "results" / "reference_based_estimation.csv"

GT_PATH, BASELINE_DIR, REFERENCE_PATH

## Load and Validate Size Data

In [ ]:
# Fail early with clear messages if result files have not been generated yet.
def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def require_dir(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required directory: {path}")

require_file(GT_PATH)
require_dir(BASELINE_DIR)
require_file(REFERENCE_PATH)

gt = pd.read_csv(GT_PATH)
reference_raw = pd.read_csv(REFERENCE_PATH)

print(f"Ground-truth rows: {len(gt)}")
print(f"Reference-based rows: {len(reference_raw)}")
print(f"Baseline JSON files: {len(list(BASELINE_DIR.glob('*.json')))}")

In [ ]:
def photo_id_from_baseline_path(path: Path) -> str:
    return re.sub(r"_vlm_baseline$", "", path.stem)

def orient_dimensions(length_like: float, width_like: float) -> tuple[float, float]:
    """Return dimensions as (longer_side, shorter_side)."""
    longer = max(float(length_like), float(width_like))
    shorter = min(float(length_like), float(width_like))
    return longer, shorter

def extract_length_width(payload: dict, source_path: Path) -> tuple[float, float]:
    # Read whichever dimension names are present, then normalize orientation below.
    length_keys = ["length_cm", "height_cm", "Length (cm)", "height"]
    width_keys = ["width_cm", "Width (cm)", "width"]

    length = next((payload[k] for k in length_keys if k in payload), None)
    width = next((payload[k] for k in width_keys if k in payload), None)
    if length is None or width is None:
        raise ValueError(f"Could not find length/width keys in {source_path}: {payload}")
    return orient_dimensions(length, width)

# Convert one JSON file per image into a pandas dataframe keyed by Photo ID.
baseline_rows = []
for path in sorted(BASELINE_DIR.glob("*.json")):
    with path.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    length_cm, width_cm = extract_length_width(payload, path)
    baseline_rows.append({
        "Photo ID": photo_id_from_baseline_path(path),
        "baseline_length_cm": length_cm,
        "baseline_width_cm": width_cm,
        "baseline_file": str(path.relative_to(REPO_ROOT)),
    })

baseline = pd.DataFrame(baseline_rows)

# Check that the reference CSV has the expected columns and rename them to match the baseline and ground-truth conventions.
reference = reference_raw.rename(columns={
    "Length (cm)": "reference_length_cm",
    "Width (cm)": "reference_width_cm",
})

required_reference_cols = {"Photo ID", "reference_length_cm", "reference_width_cm"}
missing_reference_cols = required_reference_cols - set(reference.columns)
if missing_reference_cols:
    raise ValueError(f"Reference CSV is missing columns: {sorted(missing_reference_cols)}")

gt_eval = gt.rename(columns={
    "Target Length (cm)": "truth_length_cm",
    "Target Width (cm)": "truth_width_cm",
})


# Normalize column names so baseline, reference, and ground-truth values follow one convention.
def normalize_dimension_columns(df: pd.DataFrame, prefix: str) -> None:
    """Make <prefix>_length_cm the longer side and <prefix>_width_cm the shorter side."""
    length_col = f"{prefix}_length_cm"
    width_col = f"{prefix}_width_cm"
    numeric = df[[length_col, width_col]].apply(pd.to_numeric, errors="coerce")
    df[length_col] = numeric.max(axis=1)
    df[width_col] = numeric.min(axis=1)

# Evaluate dimensions independent of axis naming: longer side is length, shorter side is width.
normalize_dimension_columns(gt_eval, "truth")
normalize_dimension_columns(reference, "reference")
normalize_dimension_columns(baseline, "baseline")

# Use left joins from the ground-truth table so missing model outputs are easy to diagnose.
data = (
    gt_eval
    .merge(baseline, on="Photo ID", how="left", validate="one_to_one")
    .merge(reference[["Photo ID", "reference_length_cm", "reference_width_cm"]], on="Photo ID", how="left", validate="one_to_one")
)

missing_baseline = data[data["baseline_length_cm"].isna()]["Photo ID"].tolist()
missing_reference = data[data["reference_length_cm"].isna()]["Photo ID"].tolist()
print(f"Joined rows: {len(data)}")
print(f"Missing baseline rows: {len(missing_baseline)}")
print(f"Missing reference rows: {len(missing_reference)}")

if missing_baseline[:10]:
    print("Example missing baseline Photo IDs:", missing_baseline[:10])
if missing_reference[:10]:
    print("Example missing reference Photo IDs:", missing_reference[:10])

# Statistical comparisons must be paired, so keep only photos with both methods available.
eval_data = data.dropna(subset=["baseline_length_cm", "baseline_width_cm", "reference_length_cm", "reference_width_cm"]).copy()
print(f"Rows usable for paired evaluation: {len(eval_data)}")
eval_data.head()

## Feature Engineering

Each photo receives categorical labels for object knowledge, object shape, camera pose, and reference placement. The notebook also computes paired error deltas. Negative deltas mean the reference-based method is better.

In [ ]:
# Collapse the CSV camera flags into the three experimental camera conditions.
def camera_pose(row: pd.Series) -> str:
    if row.get("Tilted (Y/N)") == "Y":
        return "tilted"
    if row.get("In-plane Rotation (Y/N)") == "Y":
        return "rotated"
    return "topdown"

# Rename and engineer metadata columns for stratified analysis.
eval_data["dimension_knownness"] = ["Target Dimension"].str.lower()
eval_data["shape"] = eval_data["Target Shape"].str.lower()
eval_data["camera_pose"] = eval_data.apply(camera_pose, axis=1)
eval_data["reference_placement"] = eval_data["Reference Placement"].str.lower()
eval_data["object_group"] = eval_data["Object Group"]
eval_data["shape_x_camera"] = eval_data["shape"] + " + " + eval_data["camera_pose"]
eval_data["knownness_x_camera"] = eval_data["dimension_knownness"] + " + " + eval_data["camera_pose"]
eval_data["shape_x_reference"] = eval_data["shape"] + " + ref " + eval_data["reference_placement"]

# Compute both absolute error in centimeters and scale-normalized percentage error.
for method in ["baseline", "reference"]:
    eval_data[f"{method}_length_error_cm"] = eval_data[f"{method}_length_cm"] - eval_data["truth_length_cm"]
    eval_data[f"{method}_width_error_cm"] = eval_data[f"{method}_width_cm"] - eval_data["truth_width_cm"]
    eval_data[f"{method}_length_abs_error_cm"] = eval_data[f"{method}_length_error_cm"].abs()
    eval_data[f"{method}_width_abs_error_cm"] = eval_data[f"{method}_width_error_cm"].abs()
    eval_data[f"{method}_length_ape_pct"] = 100 * eval_data[f"{method}_length_abs_error_cm"] / eval_data["truth_length_cm"]
    eval_data[f"{method}_width_ape_pct"] = 100 * eval_data[f"{method}_width_abs_error_cm"] / eval_data["truth_width_cm"]
    eval_data[f"{method}_mean_abs_error_cm"] = eval_data[[f"{method}_length_abs_error_cm", f"{method}_width_abs_error_cm"]].mean(axis=1)
    eval_data[f"{method}_mean_ape_pct"] = eval_data[[f"{method}_length_ape_pct", f"{method}_width_ape_pct"]].mean(axis=1)
    eval_data[f"{method}_max_ape_pct"] = eval_data[[f"{method}_length_ape_pct", f"{method}_width_ape_pct"]].max(axis=1)

# Delta is reference minus baseline: negative means reference-based estimation is better.
eval_data["delta_mean_abs_error_cm"] = eval_data["reference_mean_abs_error_cm"] - eval_data["baseline_mean_abs_error_cm"]
eval_data["delta_mean_ape_pct"] = eval_data["reference_mean_ape_pct"] - eval_data["baseline_mean_ape_pct"]
eval_data["reference_better_abs_error"] = eval_data["delta_mean_abs_error_cm"] < 0
eval_data["reference_better_ape"] = eval_data["delta_mean_ape_pct"] < 0

eval_data[[
    "Photo ID", "Target Name", "dimension_knownness", "shape", "camera_pose", "reference_placement",
    "truth_length_cm", "truth_width_cm", "baseline_length_cm", "baseline_width_cm", "reference_length_cm", "reference_width_cm",
    "baseline_mean_ape_pct", "reference_mean_ape_pct", "delta_mean_ape_pct"
]].head()

## Overall Paired Comparison

The same images are evaluated by both methods, so the primary analysis is paired. Negative delta values indicate lower reference-based error.

In [ ]:
# Bootstrap the mean paired delta because category subsets can be small.
def bootstrap_ci(values: pd.Series, n_boot: int = 10000, seed: int = 543) -> tuple[float, float]:
    values = values.dropna().to_numpy(dtype=float)
    if len(values) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = rng.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    return tuple(np.percentile(means, [2.5, 97.5]))

# Paired tests answer whether the mean/median per-photo delta differs from zero.
def paired_tests(delta: pd.Series) -> pd.Series:
    delta = delta.dropna().astype(float)
    out = {
        "n": len(delta),
        "mean_delta": delta.mean(),
        "median_delta": delta.median(),
        "std_delta": delta.std(ddof=1),
        "cohen_dz": delta.mean() / delta.std(ddof=1) if delta.std(ddof=1) > 0 else np.nan,
        "reference_better_rate": (delta < 0).mean(),
    }
    out["mean_delta_ci_low"], out["mean_delta_ci_high"] = bootstrap_ci(delta)

    if stats is not None and len(delta) >= 2:
        out["paired_t_p"] = stats.ttest_1samp(delta, 0.0, nan_policy="omit").pvalue
        try:
            out["wilcoxon_p"] = stats.wilcoxon(delta).pvalue
        except ValueError:
            out["wilcoxon_p"] = np.nan
    else:
        out["paired_t_p"] = np.nan
        out["wilcoxon_p"] = np.nan
    return pd.Series(out)

overall = pd.DataFrame({
    "mean_abs_error_cm": paired_tests(eval_data["delta_mean_abs_error_cm"]),
    "mean_ape_pct": paired_tests(eval_data["delta_mean_ape_pct"]),
}).T

method_summary = pd.DataFrame({
    "baseline_mean_abs_error_cm": [eval_data["baseline_mean_abs_error_cm"].mean()],
    "reference_mean_abs_error_cm": [eval_data["reference_mean_abs_error_cm"].mean()],
    "baseline_mean_ape_pct": [eval_data["baseline_mean_ape_pct"].mean()],
    "reference_mean_ape_pct": [eval_data["reference_mean_ape_pct"].mean()],
})

display(method_summary.round(3))
display(overall.round(4))

In [ ]:
# Visualize the paired improvement distribution; mass left of zero favors reference-based estimation.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(eval_data["delta_mean_abs_error_cm"], bins=20, edgecolor="white")
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set_title("Delta mean absolute error")
axes[0].set_xlabel("Reference error - baseline error (cm)")
axes[0].set_ylabel("Photos")

axes[1].hist(eval_data["delta_mean_ape_pct"], bins=20, edgecolor="white")
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_title("Delta mean absolute percentage error")
axes[1].set_xlabel("Reference APE - baseline APE (percentage points)")
axes[1].set_ylabel("Photos")

plt.tight_layout()

## Category-Level Results

The tables below show where reference-based estimation helps or hurts. `mean_delta_* < 0` means the reference method has lower error in that category.

In [ ]:
# Apply the same paired comparison inside each category for apples-to-apples subgroup analysis.
def category_summary(df: pd.DataFrame, category: str) -> pd.DataFrame:
    rows = []
    for value, group in df.groupby(category, dropna=False):
        test_abs = paired_tests(group["delta_mean_abs_error_cm"])
        test_ape = paired_tests(group["delta_mean_ape_pct"])
        rows.append({
            "category": category,
            "value": value,
            "n": len(group),
            "baseline_mae_cm": group["baseline_mean_abs_error_cm"].mean(),
            "reference_mae_cm": group["reference_mean_abs_error_cm"].mean(),
            "mean_delta_mae_cm": test_abs["mean_delta"],
            "median_delta_mae_cm": test_abs["median_delta"],
            "delta_mae_ci_low": test_abs["mean_delta_ci_low"],
            "delta_mae_ci_high": test_abs["mean_delta_ci_high"],
            "baseline_mape_pct": group["baseline_mean_ape_pct"].mean(),
            "reference_mape_pct": group["reference_mean_ape_pct"].mean(),
            "mean_delta_mape_pct": test_ape["mean_delta"],
            "reference_better_rate": test_ape["reference_better_rate"],
            "paired_t_p_mape": test_ape["paired_t_p"],
            "wilcoxon_p_mape": test_ape["wilcoxon_p"],
        })
    return pd.DataFrame(rows).sort_values(["category", "mean_delta_mape_pct"])

primary_categories = [
    "dimension_knownness",
    "shape",
    "camera_pose",
    "reference_placement",
    "object_group",
]

# Adjust subgroup p-values to reduce false discoveries from many category tests.
def benjamini_hochberg(pvalues: pd.Series) -> pd.Series:
    p = pvalues.astype(float)
    adjusted = pd.Series(np.nan, index=p.index, dtype=float)
    valid = p.dropna().sort_values()
    m = len(valid)
    if m == 0:
        return adjusted
    ranked = valid * m / np.arange(1, m + 1)
    ranked = ranked.iloc[::-1].cummin().iloc[::-1].clip(upper=1.0)
    adjusted.loc[ranked.index] = ranked
    return adjusted

primary_summary = pd.concat([category_summary(eval_data, c) for c in primary_categories], ignore_index=True)
primary_summary["wilcoxon_q_mape"] = benjamini_hochberg(primary_summary["wilcoxon_p_mape"])
display(primary_summary.round(4))

In [ ]:
# Interaction categories test whether combinations such as irregular + tilted are especially hard.
interaction_categories = [
    "shape_x_camera",
    "knownness_x_camera",
    "shape_x_reference",
]

interaction_summary = pd.concat([category_summary(eval_data, c) for c in interaction_categories], ignore_index=True)
interaction_summary["wilcoxon_q_mape"] = benjamini_hochberg(interaction_summary["wilcoxon_p_mape"])
display(interaction_summary.round(4))

In [ ]:
# Green bars indicate categories where reference-based estimation lowers MAPE on average.
plot_df = primary_summary.copy()
plot_df["label"] = plot_df["category"] + ": " + plot_df["value"].astype(str)
plot_df = plot_df.sort_values("mean_delta_mape_pct")

fig, ax = plt.subplots(figsize=(10, max(5, 0.35 * len(plot_df))))
colors = np.where(plot_df["mean_delta_mape_pct"] < 0, "#2e7d32", "#c62828")
ax.barh(plot_df["label"], plot_df["mean_delta_mape_pct"], color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Mean delta MAPE, reference - baseline (percentage points)")
ax.set_title("Reference-based advantage by primary category")
plt.tight_layout()

## Dimension-Specific Analysis

This section checks whether one method is stronger for length than width.

In [ ]:
# Analyze length and width separately in case one dimension drives the overall result.
dimension_rows = []
for dim in ["length", "width"]:
    delta_abs = eval_data[f"reference_{dim}_abs_error_cm"] - eval_data[f"baseline_{dim}_abs_error_cm"]
    delta_ape = eval_data[f"reference_{dim}_ape_pct"] - eval_data[f"baseline_{dim}_ape_pct"]
    test_abs = paired_tests(delta_abs)
    test_ape = paired_tests(delta_ape)
    dimension_rows.append({
        "dimension": dim,
        "baseline_mae_cm": eval_data[f"baseline_{dim}_abs_error_cm"].mean(),
        "reference_mae_cm": eval_data[f"reference_{dim}_abs_error_cm"].mean(),
        "mean_delta_mae_cm": test_abs["mean_delta"],
        "baseline_mape_pct": eval_data[f"baseline_{dim}_ape_pct"].mean(),
        "reference_mape_pct": eval_data[f"reference_{dim}_ape_pct"].mean(),
        "mean_delta_mape_pct": test_ape["mean_delta"],
        "reference_better_rate": test_ape["reference_better_rate"],
        "paired_t_p_mape": test_ape["paired_t_p"],
        "wilcoxon_p_mape": test_ape["wilcoxon_p"],
    })

dimension_summary = pd.DataFrame(dimension_rows)
display(dimension_summary.round(4))

## Outliers and Failure Cases

The largest positive deltas are cases where reference-based estimation is much worse. The largest negative deltas are cases where it is much better.

In [ ]:
# Inspect individual failure cases after aggregate statistics to identify practical error modes.
cols = [
    "Photo ID", "Target Name", "dimension_knownness", "shape", "camera_pose", "reference_placement",
    "truth_length_cm", "truth_width_cm",
    "baseline_length_cm", "baseline_width_cm", "baseline_mean_ape_pct",
    "reference_length_cm", "reference_width_cm", "reference_mean_ape_pct",
    "delta_mean_ape_pct",
]

print("Reference-based much worse than baseline:")
display(eval_data.sort_values("delta_mean_ape_pct", ascending=False)[cols].head(10).round(3))

print("Reference-based much better than baseline:")
display(eval_data.sort_values("delta_mean_ape_pct", ascending=True)[cols].head(10).round(3))

## Export Tables


In [ ]:
# Save clean CSV tables for the report and for any follow-up plotting outside the notebook.
EVAL_OUT_DIR = REPO_ROOT / "results" / "evaluation"
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

eval_data.to_csv(EVAL_OUT_DIR / "paired_photo_errors.csv", index=False)
overall.to_csv(EVAL_OUT_DIR / "overall_paired_tests.csv")
primary_summary.to_csv(EVAL_OUT_DIR / "primary_category_summary.csv", index=False)
interaction_summary.to_csv(EVAL_OUT_DIR / "interaction_category_summary.csv", index=False)
dimension_summary.to_csv(EVAL_OUT_DIR / "dimension_summary.csv", index=False)

print(f"Wrote evaluation tables to {EVAL_OUT_DIR.relative_to(REPO_ROOT)}")

## Report Notes Template
1. Overall: report whether the reference-based method has lower mean absolute error and lower mean absolute percentage error than the VLM baseline.
2. Significance: report the paired t-test and Wilcoxon p-values for `delta_mean_ape_pct`.
3. Practical effect: report the mean delta and its bootstrap 95% CI. A negative interval fully below zero supports reference-based improvement.
4. Category effects: discuss categories where reference-based estimation helps most and categories where it fails or is less stable.
5. Failure cases: inspect the top outliers and relate them to object shape, camera pose, and reference placement.